In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

nav_df = pd.read_csv("../data/raw/02_nav_history.csv")

nav_df["date"] = pd.to_datetime(nav_df["date"])

nav_df = nav_df.sort_values(["amfi_code", "date"]).reset_index(drop=True)

print("Shape:", nav_df.shape)
print("Columns:", nav_df.columns.tolist())
print(nav_df.head())

Shape: (46000, 3)
Columns: ['amfi_code', 'date', 'nav']
   amfi_code       date       nav
0     100016 2022-01-03  520.4608
1     100016 2022-01-04  515.0971
2     100016 2022-01-05  521.7239
3     100016 2022-01-06  515.7880
4     100016 2022-01-07  515.1639


In [2]:

nav_df["daily_return"] = (
    nav_df.groupby("amfi_code")["nav"]
    .pct_change()
)

print(nav_df[["amfi_code", "date", "nav", "daily_return"]].head(10))

   amfi_code       date       nav  daily_return
0     100016 2022-01-03  520.4608           NaN
1     100016 2022-01-04  515.0971     -0.010306
2     100016 2022-01-05  521.7239      0.012865
3     100016 2022-01-06  515.7880     -0.011377
4     100016 2022-01-07  515.1639     -0.001210
5     100016 2022-01-10  510.7136     -0.008639
6     100016 2022-01-11  513.5542      0.005562
7     100016 2022-01-12  512.3195     -0.002404
8     100016 2022-01-13  510.2445     -0.004050
9     100016 2022-01-14  514.3636      0.008073


In [3]:
annualised_returns = (
    nav_df.dropna(subset=["daily_return"])
    .groupby("amfi_code")["daily_return"]
    .agg(
        n="count",
        cumulative_growth=lambda x: (1 + x).prod()
    )
    .reset_index()
)

annualised_returns["annualised_return"] = (
    annualised_returns["cumulative_growth"]
    ** (252 / annualised_returns["n"])
    - 1
)

annualised_returns = annualised_returns[
    ["amfi_code", "n", "annualised_return"]
]

print(annualised_returns)

    amfi_code     n  annualised_return
0      100016  1149           0.025435
1      100025  1149           0.042987
2      100033  1149           0.289279
3      101206  1149           0.226265
4      101207  1149           0.076502
5      101208  1149           0.062739
6      102885  1149           0.175404
7      102886  1149           0.011304
8      102887  1149           0.162055
9      118632  1149           0.231161
10     118633  1149           0.156614
11     118634  1149           0.157567
12     118635  1149           0.153571
13     118636  1149           0.051259
14     119092  1149           0.061381
15     119093  1149           0.076105
16     119094  1149           0.271025
17     119095  1149           0.014678
18     119120  1149           0.056773
19     119551  1149           0.247966
20     119552  1149           0.206968
21     119598  1149           0.311266
22     119599  1149           0.019818
23     120503  1149           0.180708
24     120504  1149      

In [5]:

returns_df = nav_df.merge(
    annualised_returns[["amfi_code", "annualised_return"]],
    on="amfi_code",
    how="left"
)

print("returns_df created successfully")
print("Shape:", returns_df.shape)
print(returns_df.head())

returns_df created successfully
Shape: (46000, 5)
   amfi_code       date       nav  daily_return  annualised_return
0     100016 2022-01-03  520.4608           NaN           0.025435
1     100016 2022-01-04  515.0971     -0.010306           0.025435
2     100016 2022-01-05  521.7239      0.012865           0.025435
3     100016 2022-01-06  515.7880     -0.011377           0.025435
4     100016 2022-01-07  515.1639     -0.001210           0.025435


In [6]:
from pathlib import Path

output_path = Path("../data/processed/returns_computed.csv")

output_path.parent.mkdir(parents=True, exist_ok=True)

returns_df.to_csv(output_path, index=False)

print(f"Saved successfully: {output_path}")
print("Shape:", returns_df.shape)

Saved successfully: ..\data\processed\returns_computed.csv
Shape: (46000, 5)


# Day 4 - task 2 - CAGR Calculation

In [7]:

funds = (
    nav_df[["amfi_code"]]
    .drop_duplicates()
    .sort_values("amfi_code")
)

print("Number of funds:", len(funds))
print(funds.to_string(index=False))

Number of funds: 40
 amfi_code
    100016
    100025
    100033
    101206
    101207
    101208
    102885
    102886
    102887
    118632
    118633
    118634
    118635
    118636
    119092
    119093
    119094
    119095
    119120
    119551
    119552
    119598
    119599
    120503
    120504
    120505
    120506
    120507
    120841
    120842
    120843
    120844
    125497
    125498
    148567
    148568
    148569
    149322
    149323
    149324


In [8]:

nav_df["date"] = pd.to_datetime(nav_df["date"])

periods = {
    "1Y": 1,
    "3Y": 3,
    "5Y": 5
}

cagr_results = []

for amfi_code, fund_data in nav_df.groupby("amfi_code"):

    fund_data = fund_data.sort_values("date").dropna(subset=["nav"])

    end_date = fund_data["date"].max()
    end_nav = fund_data.loc[
        fund_data["date"] == end_date, "nav"
    ].iloc[0]

    for period_name, years in periods.items():

        target_start_date = end_date - pd.DateOffset(years=years)

        eligible = fund_data[
            fund_data["date"] <= target_start_date
        ]

        if eligible.empty:
            continue

        start_row = eligible.iloc[-1]

        start_date = start_row["date"]
        start_nav = start_row["nav"]

        # CAGR formula
        cagr = (end_nav / start_nav) ** (1 / years) - 1

        cagr_results.append({
            "amfi_code": amfi_code,
            "period": period_name,
            "years": years,
            "start_date": start_date,
            "end_date": end_date,
            "start_nav": start_nav,
            "end_nav": end_nav,
            "cagr": cagr
        })

cagr_report = pd.DataFrame(cagr_results)

print("Shape:", cagr_report.shape)
print(cagr_report.head(15))

Shape: (80, 8)
    amfi_code period  years start_date   end_date  start_nav   end_nav  \
0      100016     1Y      1 2025-05-29 2026-05-29   596.8877  583.6113   
1      100016     3Y      3 2023-05-29 2026-05-29   561.5519  583.6113   
2      100025     1Y      1 2025-05-29 2026-05-29    30.7452   31.8843   
3      100025     3Y      3 2023-05-29 2026-05-29    28.4135   31.8843   
4      100033     1Y      1 2025-05-29 2026-05-29   223.1951  342.0072   
5      100033     3Y      3 2023-05-29 2026-05-29   147.2155  342.0072   
6      101206     1Y      1 2025-05-29 2026-05-29   522.7639  773.2939   
7      101206     3Y      3 2023-05-29 2026-05-29   360.4971  773.2939   
8      101207     1Y      1 2025-05-29 2026-05-29    71.0180   53.9836   
9      101207     3Y      3 2023-05-29 2026-05-29    61.3081   53.9836   
10     101208     1Y      1 2025-05-29 2026-05-29   382.4272  410.1021   
11     101208     3Y      3 2023-05-29 2026-05-29   341.2705  410.1021   
12     102885     1Y   

In [9]:
cagr_report["cagr_pct"] = cagr_report["cagr"] * 100

print(
    cagr_report[
        ["amfi_code", "period", "start_date", "end_date",
         "start_nav", "end_nav", "cagr_pct"]
    ].to_string(index=False)
)

 amfi_code period start_date   end_date  start_nav   end_nav   cagr_pct
    100016     1Y 2025-05-29 2026-05-29   596.8877  583.6113  -2.224271
    100016     3Y 2023-05-29 2026-05-29   561.5519  583.6113   1.292649
    100025     1Y 2025-05-29 2026-05-29    30.7452   31.8843   3.704969
    100025     3Y 2023-05-29 2026-05-29    28.4135   31.8843   3.916390
    100033     1Y 2025-05-29 2026-05-29   223.1951  342.0072  53.232396
    100033     3Y 2023-05-29 2026-05-29   147.2155  342.0072  32.442459
    101206     1Y 2025-05-29 2026-05-29   522.7639  773.2939  47.924120
    101206     3Y 2023-05-29 2026-05-29   360.4971  773.2939  28.967695
    101207     1Y 2025-05-29 2026-05-29    71.0180   53.9836 -23.986032
    101207     3Y 2023-05-29 2026-05-29    61.3081   53.9836  -4.152381
    101208     1Y 2025-05-29 2026-05-29   382.4272  410.1021   7.236645
    101208     3Y 2023-05-29 2026-05-29   341.2705  410.1021   6.315784
    102885     1Y 2025-05-29 2026-05-29   156.2127  187.7797  20

In [10]:
from pathlib import Path

output_path = Path("../data/processed/cagr_report.csv")

output_path.parent.mkdir(parents=True, exist_ok=True)

cagr_report.to_csv(output_path, index=False)

print(f"Saved successfully: {output_path}")
print("Shape:", cagr_report.shape)

Saved successfully: ..\data\processed\cagr_report.csv
Shape: (80, 9)


In [11]:
check = pd.read_csv(output_path)

print(check.shape)
print(check.head())
print("\nPeriods:")
print(check["period"].value_counts())

(80, 9)
   amfi_code period  years  start_date    end_date  start_nav   end_nav  \
0     100016     1Y      1  2025-05-29  2026-05-29   596.8877  583.6113   
1     100016     3Y      3  2023-05-29  2026-05-29   561.5519  583.6113   
2     100025     1Y      1  2025-05-29  2026-05-29    30.7452   31.8843   
3     100025     3Y      3  2023-05-29  2026-05-29    28.4135   31.8843   
4     100033     1Y      1  2025-05-29  2026-05-29   223.1951  342.0072   

       cagr   cagr_pct  
0 -0.022243  -2.224271  
1  0.012926   1.292649  
2  0.037050   3.704969  
3  0.039164   3.916390  
4  0.532324  53.232396  

Periods:
period
1Y    40
3Y    40
Name: count, dtype: int64


# Day 4 — Task 3: Sharpe Ratio 

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

returns_path = Path("../data/processed/returns_computed.csv")

returns_df = pd.read_csv(returns_path)

returns_df["date"] = pd.to_datetime(returns_df["date"])

print("returns_df loaded successfully")
print("Shape:", returns_df.shape)
returns_df.head()

returns_df loaded successfully
Shape: (46000, 5)


,amfi_code,date,nav,daily_return,annualised_return
0,100016,2022-01-03,520.4608,NaN,0.025435
1,100016,2022-01-04,515.0971,-0.010306,0.025435
2,100016,2022-01-05,521.7239,0.012865,0.025435
3,100016,2022-01-06,515.7880,-0.011377,0.025435
4,100016,2022-01-07,515.1639,-0.001210,0.025435


In [3]:
RF = 0.065

volatility = (
    returns_df.groupby("amfi_code")["daily_return"]
    .std()
    .reset_index(name="daily_std")
)

avg_return = (
    returns_df.groupby("amfi_code")["daily_return"]
    .mean()
    .reset_index(name="daily_mean")
)

sharpe_df = avg_return.merge(volatility, on="amfi_code")

sharpe_df["annualised_return"] = (
    (1 + sharpe_df["daily_mean"]) ** 252 - 1
)

sharpe_df["annualised_volatility"] = (
    sharpe_df["daily_std"] * np.sqrt(252)
)

sharpe_df["sharpe_ratio"] = (
    (sharpe_df["annualised_return"] - RF)
    / sharpe_df["annualised_volatility"]
)

sharpe_df.head()

,amfi_code,daily_mean,daily_std,annualised_return,annualised_volatility,sharpe_ratio
0,100016,0.000142,0.009164,0.036325,0.145481,-0.197106
1,100025,0.000170,0.002460,0.043781,0.039052,-0.543340
2,100033,0.001080,0.011929,0.312539,0.189367,1.307193
3,101206,0.000852,0.009177,0.239311,0.145682,1.196513
4,101207,0.000424,0.016251,0.112867,0.257973,0.185550


In [4]:
output_path = Path("../data/processed/sharpe_values.csv")

output_path.parent.mkdir(parents=True, exist_ok=True)

sharpe_df.to_csv(output_path, index=False)

print("Saved successfully:", output_path)
print("Shape:", sharpe_df.shape)

Saved successfully: ..\data\processed\sharpe_values.csv
Shape: (40, 6)
